In [1]:
!pip install --no-index --find-links /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages segmentation-models-pytorch albumentations omegaconf torchmetrics

Looking in links: /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages
Processing /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages/segmentation_models_pytorch-0.5.0-py3-none-any.whl


In [2]:
import os
from pathlib import Path

SUNNY_DIRS = [
    Path('/kaggle/input/datasets/leehien12/uav-forest-sunny-part1'),
    Path('/kaggle/input/datasets/khesng/uav-forest-sunny-next'),
    Path('/kaggle/input/datasets/leehien12/uav-forest-sunny'),
]

# Chia THEO SEQUENCE (không theo frame) — tránh lặp lại lỗi leakage
NADIR_SPLIT_CONFIG = {
    'train': ['seq3', 'seq9'],   # 30m + 80m, pitch=90°
    'val':   ['seq6'],           # 50m, pitch=90° — val riêng cho pretrain này
    'test':  [],
}

def collect_pairs_from_dirs(base_dirs, split_config):
    data = {'train': [], 'val': [], 'test': []}
    found_sequences = {}
    all_dirs = base_dirs if isinstance(base_dirs, list) else [base_dirs]

    for base_path in all_dirs:
        if not Path(base_path).exists():
            print(f"  ⚠️  Không tìm thấy: {base_path}")
            continue
        for color_dir in Path(base_path).rglob('color'):
            if not color_dir.is_dir():
                continue
            seq_dir  = color_dir.parent
            seq_name = seq_dir.name

            mask_dir = seq_dir / 'labels'
            if not mask_dir.exists():
                mask_dir = seq_dir / 'label'
            if not mask_dir.exists():
                continue

            target_set = None
            for key, seq_list in split_config.items():
                if seq_name in seq_list:
                    target_set = key
                    break
            if not target_set:
                continue  # sequence không nằm trong seq3/6/9 → bỏ qua

            n_added = 0
            for img_p in sorted(list(color_dir.glob('*.png')) + list(color_dir.glob('*.jpg'))):
                mask_p = (mask_dir / img_p.name).with_suffix('.png')
                if not mask_p.exists():
                    mask_p = (mask_dir / img_p.name).with_suffix('.jpg')
                if mask_p.exists():
                    data[target_set].append((str(img_p), str(mask_p)))
                    n_added += 1

            found_sequences[seq_name] = {'set': target_set, 'count': n_added,
                                          'source': Path(base_path).name}
    return data, found_sequences

print("=" * 60)
print("  THU THẬP DATA NADIR-ONLY: seq3, seq6, seq9")
print("=" * 60)

nadir_data, nadir_found_seqs = collect_pairs_from_dirs(SUNNY_DIRS, NADIR_SPLIT_CONFIG)
train_files = nadir_data['train']
val_files   = nadir_data['val']

expected = set(NADIR_SPLIT_CONFIG['train'] + NADIR_SPLIT_CONFIG['val'])
missing  = expected - set(nadir_found_seqs.keys())
if missing:
    print(f"\n  ❌ THIẾU sequences: {sorted(missing)} — kiểm tra lại tên thư mục.\n")
else:
    print("\n  ✅ Tìm thấy đủ seq3, seq6, seq9.")

for seq, info in sorted(nadir_found_seqs.items()):
    print(f"    {seq:8s} → {info['set']:6s} | {info['count']:5,} ảnh | [{info['source']}]")

print(f"\n  ✅ TRAIN : {len(train_files):6,} cặp (seq3+seq9)")
print(f"  ✅ VAL   : {len(val_files):6,} cặp (seq6)")

if len(train_files) == 0 or len(val_files) == 0:
    raise ValueError("Thiếu data! Kiểm tra lại SUNNY_DIRS.")

  THU THẬP DATA NADIR-ONLY: seq3, seq6, seq9

  ✅ Tìm thấy đủ seq3, seq6, seq9.
    seq3     → train  | 1,469 ảnh | [uav-forest-sunny-part1]
    seq6     → val    | 1,437 ảnh | [uav-forest-sunny-next]
    seq9     → train  | 1,397 ảnh | [uav-forest-sunny]

  ✅ TRAIN :  2,866 cặp (seq3+seq9)
  ✅ VAL   :  1,437 cặp (seq6)


In [4]:
import os
import random
import numpy as np
import torch

def seed_everything(seed=2025):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 123
seed_everything(SEED)

MODEL_NAME      = 'unetpp'   
ENCODER         = 'resnet50'
ENCODER_WEIGHTS = 'imagenet'
IMG_SIZE        = (768, 1280)
BATCH_SIZE      = 4
EPOCHS          = 50
LR              = 1e-4
NUM_WORKERS     = 2
PATIENCE        = 10
USE_AMP         = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[INFO] Device: {DEVICE} | IMG_SIZE: {IMG_SIZE} | Model: {MODEL_NAME}")

[INFO] Device: cuda | IMG_SIZE: (768, 1280) | Model: unetpp


In [5]:
import numpy as np
import random
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

LABEL_COLORS = [
    (0, 255, 255),    # 0: Sky
    (0, 127, 0),      # 1: Deciduous_trees
    (19, 132, 69),    # 2: Coniferous_trees
    (0, 53, 65),      # 3: Fallen_trees
    (130, 76, 0),     # 4: Dirt_ground
    (152, 251, 152),  # 5: Ground_vegetation
    (151, 126, 171),  # 6: Rocks
    (250, 150, 0),    # 7: Building
    (115, 176, 195),  # 8: Fence
    (123, 123, 123),  # 9: Car
    (0, 0, 0),        # 10: Empty
]
CLASS_NAMES = [
    'Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
    'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
    'Fence', 'Car', 'Empty',
]
NUM_CLASSES = len(CLASS_NAMES)  # = 11

RGB_TO_CLASS_LUT = np.full((256, 256, 256), NUM_CLASSES - 1, dtype=np.uint8)
for cid, color in enumerate(LABEL_COLORS):
    RGB_TO_CLASS_LUT[color[0], color[1], color[2]] = cid

def fast_rgb_to_mask(rgb_arr):
    return RGB_TO_CLASS_LUT[rgb_arr[..., 0], rgb_arr[..., 1], rgb_arr[..., 2]]

class ForestDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path, mask_path = self.file_list[idx]
        img      = np.array(Image.open(img_path).convert('RGB'))
        mask_rgb = np.array(Image.open(mask_path).convert('RGB'))
        mask     = fast_rgb_to_mask(mask_rgb)
        if self.transform:
            aug       = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

H, W = IMG_SIZE   # (768, 1280) — lấy từ config đã chạy ở cell 3
EMPTY_CLASS = NUM_CLASSES - 1  # = 10

train_tf = A.Compose([
    A.Resize(height=H, width=W),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), rotate=(-15, 15),
              translate_percent={"x": (-0.0625, 0.0625), "y": (-0.0625, 0.0625)},
              p=0.3, border_mode=0, fill=0, fill_mask=EMPTY_CLASS),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(height=H, width=W),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_ds = ForestDataset(train_files, train_tf)
val_ds   = ForestDataset(val_files,   val_tf)

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    worker_init_fn=worker_init_fn,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"[OK] Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")

[OK] Train: 716 batches | Val: 360 batches


In [6]:
from pathlib import Path
import os
import torch

# ---- OUTPUT_DIR (thiếu, gây lỗi vừa rồi) ----
IS_KAGGLE  = os.path.exists('/kaggle')
OUTPUT_DIR = Path('/kaggle/working/outputs') if IS_KAGGLE else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- CheckpointSaver (chưa định nghĩa, sẽ lỗi tiếp nếu không thêm) ----
class CheckpointSaver:
    def __init__(self, dirpath, maximize=True):
        self.dirpath    = dirpath
        self.maximize   = maximize
        self.best_score = None
        self.best_path  = None

    def save_if_best(self, state_dict, epoch, current_score):
        is_better = (
            self.best_score is None or
            (self.maximize     and current_score > self.best_score) or
            (not self.maximize and current_score < self.best_score)
        )
        if is_better:
            if self.best_path and os.path.exists(self.best_path):
                if '/kaggle/input' not in self.best_path:
                    try:
                        os.remove(self.best_path)
                        print(f'  🗑️  Đã xóa checkpoint cũ: {os.path.basename(self.best_path)}')
                    except Exception as e:
                        print(f'  ⚠️  Không thể xóa checkpoint cũ. Lỗi: {e}')
                else:
                    print(f'  🛡️  Bỏ qua xóa checkpoint cũ vì thuộc thư mục Read-only (Input)')
            filename = f"epoch_{epoch:03d}_miou_{current_score:.4f}.pth"
            filepath = os.path.join(self.dirpath, filename)
            torch.save(state_dict, filepath)
            self.best_score = current_score
            self.best_path  = filepath
            print(f'  ✅ [BEST] mIoU: {current_score:.4f} → {filename}')
            return True
        return False

In [7]:
import os, shutil, glob
from tqdm import tqdm
import torch.nn as nn
import segmentation_models_pytorch as smp

def compute_class_weights(file_list, num_classes, c=1.02):
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, mask_path in tqdm(file_list, desc='Computing class weights'):
        mask_rgb = np.array(Image.open(mask_path).convert('RGB'))
        mask = fast_rgb_to_mask(mask_rgb)
        for cid in range(num_classes):
            counts[cid] += (mask == cid).sum()
    total = counts.sum()
    freq  = counts / total
    w     = 1.0 / np.log(c + freq)
    w     = w / w.sum() * num_classes
    print("\n📊 Class weights (nadir-only):")
    for name, wi, fi in zip(CLASS_NAMES, w, freq):
        print(f"  {name:<22} freq={fi*100:6.3f}%  weight={wi:.4f}")
    return torch.tensor(w, dtype=torch.float32).to(DEVICE)

class_weights = compute_class_weights(train_files, NUM_CLASSES)

# --- Model (giữ nguyên MODEL_NAME/ENCODER từ cell 3) ---
CACHE_DIR = '/root/.cache/torch/hub/checkpoints/'
os.makedirs(CACHE_DIR, exist_ok=True)
WEIGHT_DIR = '/kaggle/input/datasets/itmeagain77/resnet50-pytorch-weights'
pth_files  = glob.glob(os.path.join(WEIGHT_DIR, '*.pth'))
if pth_files:
    cache_file = os.path.join(CACHE_DIR, 'resnet50-19c8e357.pth')
    if not os.path.exists(cache_file):
        shutil.copy(pth_files[0], cache_file)

MODELS = {
    'unet':          lambda: smp.Unet(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'unetpp':        lambda: smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'deeplabv3plus': lambda: smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'segformer':     lambda: smp.Segformer(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
}
model = MODELS[MODEL_NAME]().to(DEVICE)
if torch.cuda.device_count() > 1:
    print(f"[INFO] Dùng {torch.cuda.device_count()} GPU qua DataParallel")
    model = nn.DataParallel(model)
print(f'🚀 MODEL: {MODEL_NAME} | Encoder: {ENCODER} | Classes: {NUM_CLASSES}')

# --- Adaptive Uncertainty Loss (công thức (6) trong bài báo) ---
class AdaptiveFocalDiceLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.focal = smp.losses.FocalLoss(mode=smp.losses.MULTICLASS_MODE, gamma=2.0)
        self.dice  = smp.losses.DiceLoss(mode=smp.losses.MULTICLASS_MODE, from_logits=True,
                                          classes=list(range(NUM_CLASSES - 1)))
        self.ce    = nn.CrossEntropyLoss(weight=class_weights, ignore_index=NUM_CLASSES - 1)
        self.log_sigma_focal = nn.Parameter(torch.zeros(1))
        self.log_sigma_dice  = nn.Parameter(torch.zeros(1))
        self.log_sigma_ce    = nn.Parameter(torch.zeros(1))

    def forward(self, logits, targets):
        l_focal = self.focal(logits, targets)
        l_dice  = self.dice(logits, targets)
        l_ce    = self.ce(logits, targets)
        w_focal = torch.exp(-self.log_sigma_focal)
        w_dice  = torch.exp(-self.log_sigma_dice)
        w_ce    = torch.exp(-self.log_sigma_ce)
        loss = (0.5 * w_focal * l_focal + 0.5 * self.log_sigma_focal
              + 0.5 * w_dice  * l_dice  + 0.5 * self.log_sigma_dice
              + 0.5 * w_ce    * l_ce    + 0.5 * self.log_sigma_ce)
        return loss, {
            'focal': l_focal.item(), 'dice': l_dice.item(), 'ce': l_ce.item(),
            'w_focal': w_focal.item(), 'w_dice': w_dice.item(), 'w_ce': w_ce.item(),
        }

    def get_weights(self):
        return {'w_focal': torch.exp(-self.log_sigma_focal).item(),
                'w_dice':  torch.exp(-self.log_sigma_dice).item(),
                'w_ce':    torch.exp(-self.log_sigma_ce).item()}

criterion = AdaptiveFocalDiceLoss(class_weights=class_weights).to(DEVICE)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(criterion.parameters()),
                               lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)
print(f"✅ Sẵn sàng train | Initial weights: {criterion.get_weights()}")

Computing class weights: 100%|██████████| 2866/2866 [00:48<00:00, 59.04it/s]



📊 Class weights (nadir-only):
  Sky                    freq= 0.000%  weight=1.6348
  Deciduous_trees        freq=19.228%  weight=0.1682
  Coniferous_trees       freq=21.644%  weight=0.1525
  Fallen_trees           freq= 0.887%  weight=1.1377
  Dirt_ground            freq= 3.133%  weight=0.6467
  Ground_vegetation      freq=53.023%  weight=0.0738
  Rocks                  freq= 1.783%  weight=0.8719
  Building               freq= 0.222%  weight=1.4728
  Fence                  freq= 0.010%  weight=1.6269
  Car                    freq= 0.071%  weight=1.5796
  Empty                  freq= 0.000%  weight=1.6349


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/smp-hub/resnet50.imagenet/resolve/00cb74e366966d59cd9a35af57e618af9f88efe9/config.json
Retrying in 1s [Retry 1/5].
/usr/local/lib/python3.12/dist-packages/segmentation_models_pytorch/encoders/__init__.py:136: UserWarning: Error loading resnet50 `imagenet` weights from Hugging Face Hub, trying loading from original url...
  warnings.warn(message, UserWarning)


🚀 MODEL: unetpp | Encoder: resnet50 | Classes: 11
✅ Sẵn sàng train | Initial weights: {'w_focal': 1.0, 'w_dice': 1.0, 'w_ce': 1.0}


In [8]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# Vấn đề cũ: loss = focal + dice + 0.5 * ce  (trọng số cố định)
# Cải tiến:  loss = w_focal*focal + w_dice*dice + w_ce*ce
#            trong đó w_focal, w_dice, w_ce được học tự động qua uncertainty weighting
#            (Kendall et al., 2018) — mỗi weight là log(sigma^2), sigma được tối ưu
#            cùng với model parameters
#
# Ý tưởng: nếu một thành phần loss đang "khó" (variance cao), model sẽ tự giảm
# trọng số của nó để không bị dominated; nếu một loss đang học tốt, trọng số tăng.

class AdaptiveFocalDiceLoss(nn.Module):
    """
    Kết hợp Focal Loss + Dice Loss + Weighted CE với trọng số thích nghi.

    Sử dụng Multi-Task Uncertainty Weighting (Kendall et al., 2018):
        L_total = sum_i [ (1/2*sigma_i^2) * L_i + log(sigma_i) ]

    log_sigma_i là tham số learnable — được tối ưu cùng model.
    Điều này tốt hơn fixed weighting vì:
    - Không cần tune thủ công tỷ lệ giữa các loss
    - Tự điều chỉnh theo từng batch/epoch dựa trên độ khó
    - Có thể quan sát quá trình các sigma thay đổi để hiểu behavior của model
    """
    def __init__(self, class_weights=None):
        super().__init__()

        # 3 thành phần loss
        self.focal = smp.losses.FocalLoss(
            mode=smp.losses.MULTICLASS_MODE, gamma=2.0)
        self.dice  = smp.losses.DiceLoss(
            mode=smp.losses.MULTICLASS_MODE,
            from_logits=True,
            classes=list(range(NUM_CLASSES - 1)))  # bỏ class Empty
        self.ce    = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=NUM_CLASSES - 1)

        # log(sigma^2) learnable — khởi tạo = 0 nghĩa là sigma=1, weight=0.5
        # Dùng log để đảm bảo sigma > 0 trong suốt quá trình train
        self.log_sigma_focal = nn.Parameter(torch.zeros(1))
        self.log_sigma_dice  = nn.Parameter(torch.zeros(1))
        self.log_sigma_ce    = nn.Parameter(torch.zeros(1))

    def forward(self, logits, targets):
        l_focal = self.focal(logits, targets)
        l_dice  = self.dice(logits, targets)
        l_ce    = self.ce(logits, targets)

        # Uncertainty weighting: L_i / (2*sigma_i^2) + log(sigma_i)
        # = L_i * exp(-log_sigma^2) * 0.5 + 0.5 * log_sigma^2
        w_focal = torch.exp(-self.log_sigma_focal)
        w_dice  = torch.exp(-self.log_sigma_dice)
        w_ce    = torch.exp(-self.log_sigma_ce)

        loss = (0.5 * w_focal * l_focal + 0.5 * self.log_sigma_focal
              + 0.5 * w_dice  * l_dice  + 0.5 * self.log_sigma_dice
              + 0.5 * w_ce    * l_ce    + 0.5 * self.log_sigma_ce)

        return loss, {
            'focal':       l_focal.item(),
            'dice':        l_dice.item(),
            'ce':          l_ce.item(),
            'w_focal':     w_focal.item(),
            'w_dice':      w_dice.item(),
            'w_ce':        w_ce.item(),
        }

    def get_weights(self):
        """In ra trọng số hiện tại để theo dõi quá trình thích nghi."""
        return {
            'w_focal': torch.exp(-self.log_sigma_focal).item(),
            'w_dice':  torch.exp(-self.log_sigma_dice).item(),
            'w_ce':    torch.exp(-self.log_sigma_ce).item(),
        }


criterion = AdaptiveFocalDiceLoss(class_weights=class_weights).to(DEVICE)

# Quan trọng: thêm parameters của loss vào optimizer
# để log_sigma được cập nhật cùng với model weights
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=LR, weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print("✅ AdaptiveFocalDiceLoss + Optimizer + Scheduler khởi tạo thành công!")
print(f"   Learnable loss weights: log_sigma_focal, log_sigma_dice, log_sigma_ce")
print(f"   Initial weights: {criterion.get_weights()}")

✅ AdaptiveFocalDiceLoss + Optimizer + Scheduler khởi tạo thành công!
   Learnable loss weights: log_sigma_focal, log_sigma_dice, log_sigma_ce
   Initial weights: {'w_focal': 1.0, 'w_dice': 1.0, 'w_ce': 1.0}


In [9]:
def get_miou(cm, n_ignore_last=1):
    """
    Tính mIoU từ confusion matrix.
    n_ignore_last=1  → bỏ class Empty (index 10) khỏi mIoU  ← LUÔN DÙNG GIÁ TRỊ NÀY
    n_ignore_last=0  → tính tất cả class (KHÔNG dùng cho eval chính thức)
    """
    inter = np.diag(cm)
    union = cm.sum(1) + cm.sum(0) - inter
    iou   = np.where(union > 0, inter / union, 0.0)

    valid_iou   = iou[:-n_ignore_last]   if n_ignore_last > 0 else iou
    valid_union = union[:-n_ignore_last] if n_ignore_last > 0 else union
    miou = valid_iou[valid_union > 0].mean() if any(valid_union > 0) else 0.0
    return miou, iou

print("[OK] get_miou định nghĩa với n_ignore_last=1 (bỏ class Empty khi eval)")

[OK] get_miou định nghĩa với n_ignore_last=1 (bỏ class Empty khi eval)


In [10]:
def read_split_file(txt_path):
    img_files   = []
    label_files = []

    with open(txt_path) as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    for line in lines:
        seq_name, fname = line.split('/')
        base_name = fname.split('.')[0]

        # Sửa lỗi: Tách biệt rõ ràng đường dẫn của OLD và NEW cho cả ảnh và nhãn
        if seq_name == "seq00":
            img_dir = os.path.join(WILD_UAV_OLD, "mapping_set", f"{seq_name}_img+metadata", "img")
            label_dir = os.path.join(WILD_UAV_OLD, "mapping_set", "semantic_extension", f"{seq_name}_semantic", "semantic")
        else:
            img_dir = os.path.join(WILD_UAV_NEW, f"{seq_name}_imgmetadata.7z", f"{seq_name}_img+metadata.7z", "img")
            label_dir = os.path.join(WILD_UAV_NEW, "mapping_set", "semantic_extension", f"{seq_name}_semantic", "semantic")

        # Quét thực tế trên đĩa để lấy ảnh
        png_path = os.path.join(img_dir, f"{base_name}.png")
        jpg_path = os.path.join(img_dir, f"{base_name}.jpg")
        img_path = png_path if os.path.exists(png_path) else jpg_path

        # Lấy file nhãn
        label_path = os.path.join(label_dir, f"{base_name}.png")

        img_files.append(img_path)
        label_files.append(label_path)

    return img_files, label_files

In [2]:
import os, glob, gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# =====================================================================
# 1. CẤU HÌNH CƠ BẢN
# =====================================================================
ENCODER      = "resnet50"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE   = 4
EPOCHS       = 50
LR           = 1e-4
NUM_CLASSES  = 16
IGNORE_INDEX = 255
USE_AMP      = True
PATIENCE     = 10   # Dừng sớm nếu val mIoU không cải thiện sau 10 epoch

WILD_UAV_OLD = "/kaggle/input/datasets/dinhh04/wilduav/WildUAV"
WILD_UAV_NEW = "/kaggle/input/datasets/phmnguynkimquyn/wuav-data"
WEIGHT_DIR   = "/kaggle/input/datasets/itmeagain77/resnet50-pytorch-weights"

# Split gốc của WildUAV (chỉ có train/val, KHÔNG có test.txt riêng)
TRAIN_TXT = "/kaggle/input/datasets/itmeagain77/wilduav-splits/train.txt"
VAL_TXT   = "/kaggle/input/datasets/itmeagain77/wilduav-splits/val.txt"

# =====================================================================
# 2. BẢNG MÀU, LUT & DATASET
# =====================================================================
WILDUAV_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65), (130, 76, 0),
    (152, 251, 152), (151, 126, 171), (0, 0, 255), (250, 150, 0), (115, 176, 195),
    (128, 64, 128), (255, 77, 228), (123, 123, 123), (255, 255, 255), (200, 0, 0), (0, 0, 0)
]
RGB_TO_WILDUAV_LUT = np.full((256, 256, 256), 15, dtype=np.uint8)
for cid, color in enumerate(WILDUAV_COLORS):
    RGB_TO_WILDUAV_LUT[color[0], color[1], color[2]] = cid

class WildUAVDataset(Dataset):
    def __init__(self, img_paths, label_paths, transform=None):
        self.img_paths   = img_paths
        self.label_paths = label_paths
        self.transform   = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img       = np.array(Image.open(self.img_paths[idx]).convert('RGB'))
        label_rgb = np.array(Image.open(self.label_paths[idx]).convert('RGB'))
        mask      = RGB_TO_WILDUAV_LUT[label_rgb[..., 0], label_rgb[..., 1], label_rgb[..., 2]]
        if self.transform:
            aug       = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
test_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# =====================================================================
# 3. CÁC HÀM BỔ TRỢ (Đọc Data, Lọc Data, Evaluate) — GIỮ NGUYÊN NHƯ BẢN GỐC
# =====================================================================
def read_split_file(txt_path):
    img_files, label_files = [], []
    with open(txt_path) as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    for line in lines:
        seq_name, fname = line.split('/')
        base_name = fname.split('.')[0]
        if seq_name == "seq00":
            img_dir = os.path.join(WILD_UAV_OLD, "mapping_set", f"{seq_name}_img+metadata", "img")
        else:
            img_dir = os.path.join(WILD_UAV_NEW, f"{seq_name}_imgmetadata.7z", f"{seq_name}_img+metadata.7z", "img")
        png_path = os.path.join(img_dir, f"{base_name}.png")
        jpg_path = os.path.join(img_dir, f"{base_name}.jpg")
        img_path = png_path if os.path.exists(png_path) else jpg_path
        label_path = os.path.join(WILD_UAV_OLD, "mapping_set", "semantic_extension", f"{seq_name}_semantic", "semantic", f"{base_name}.png")
        img_files.append(img_path)
        label_files.append(label_path)
    return img_files, label_files

def filter_valid_pairs(img_list, label_list):
    valid_imgs, valid_labels = [], []
    for img, lbl in zip(img_list, label_list):
        if os.path.exists(img) and os.path.exists(lbl):
            valid_imgs.append(img)
            valid_labels.append(lbl)
    return valid_imgs, valid_labels

EVAL_CLASSES = ['Ground_vegetation', 'Trees', 'Dirt_ground', 'Road', 'Water', 'Empty', 'Static_car', 'Moving_car', 'People']
NUM_EVAL = len(EVAL_CLASSES)

WILD_MAP_LUT = np.full((NUM_CLASSES,), 5, dtype=np.int64)  # default = Empty
WILD_MAP_LUT[0] = 0; WILD_MAP_LUT[5] = 0
WILD_MAP_LUT[1] = 1; WILD_MAP_LUT[2] = 1; WILD_MAP_LUT[3] = 1
WILD_MAP_LUT[4] = 2; WILD_MAP_LUT[10] = 3; WILD_MAP_LUT[11] = 3
WILD_MAP_LUT[7] = 4; WILD_MAP_LUT[15] = 5
WILD_MAP_LUT[12] = 6; WILD_MAP_LUT[13] = 7; WILD_MAP_LUT[14] = 8

def evaluate(model, loader):
    model.eval()
    cm = np.zeros((NUM_EVAL, NUM_EVAL), dtype=np.int64)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                preds = torch.argmax(model(imgs), dim=1).cpu().numpy()
            masks = masks.numpy()
            g_eval = WILD_MAP_LUT[masks]
            p_eval = WILD_MAP_LUT[preds]
            keep = (g_eval >= 0) & (g_eval < NUM_EVAL)
            cm += np.bincount(g_eval[keep] * NUM_EVAL + p_eval[keep], minlength=NUM_EVAL ** 2).reshape(NUM_EVAL, NUM_EVAL)

    inter = np.diag(cm)
    union = cm.sum(1) + cm.sum(0) - inter
    with np.errstate(divide='ignore', invalid='ignore'):
        iou = np.where(union > 0, inter / union, 0.0)
    return iou, iou.mean(), union

# =====================================================================
# 4. CHẠY 1 LẦN DUY NHẤT TRÊN SPLIT GỐC (REAL-ONLY, OFFICIAL SPLIT)
# =====================================================================
if __name__ == "__main__":
    print(f"\n{'='*60}")
    print(f"  🔥 Real-only protocol — dùng split gốc train.txt / val.txt")
    print(f"{'='*60}\n")

    train_imgs_raw, train_labels_raw = read_split_file(TRAIN_TXT)
    val_imgs_raw,   val_labels_raw   = read_split_file(VAL_TXT)

    train_imgs, train_labels = filter_valid_pairs(train_imgs_raw, train_labels_raw)
    val_imgs,   val_labels   = filter_valid_pairs(val_imgs_raw,   val_labels_raw)
    print(f"📊 Dataset: Train: {len(train_imgs)} | Val: {len(val_imgs)}")
    print("⚠️  Lưu ý: split gốc không có test.txt riêng — val.txt vừa dùng để chọn best "
          "checkpoint, vừa dùng để báo cáo mIoU cuối cùng (không có tập test độc lập).")

    train_loader = DataLoader(WildUAVDataset(train_imgs, train_labels, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(WildUAVDataset(val_imgs,   val_labels, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # Khởi tạo mô hình MỚI (chỉ Load ImageNet)
    model = smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights=None, classes=NUM_CLASSES, activation=None)
    pth_files = glob.glob(os.path.join(WEIGHT_DIR, '*.pth'))
    if pth_files:
        state_dict = torch.load(pth_files[0], map_location='cpu', weights_only=False)
        model.encoder.load_state_dict(state_dict)
    model = model.to(DEVICE)

    ce_loss   = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=IGNORE_INDEX)
    focal_loss = smp.losses.FocalLoss(mode='multiclass', ignore_index=IGNORE_INDEX)
    
    W_FOCAL = 1.0
    W_DICE = 1.0
    W_CE = 0.5
    
    def criterion(preds, targets):
        loss_ce = ce_loss(preds, targets)
        loss_dice = dice_loss(preds, targets)
        loss_focal = focal_loss(preds, targets)
        return (W_CE * loss_ce) + (W_DICE * loss_dice) + (W_FOCAL * loss_focal)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_val_miou = 0.0
    no_improve = 0
    CKPT_PATH = "/kaggle/working/best_realonly_officialsplit.pth"

    for epoch in range(EPOCHS):
        model.train()
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        scheduler.step()

        # Đánh giá trên tập VAL để lấy best_checkpoint
        _, val_miou, _ = evaluate(model, val_loader)
        print(f"Epoch {epoch+1} | Val mIoU: {val_miou*100:.1f}%")

        if val_miou > best_val_miou:
            best_val_miou = val_miou
            no_improve = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_miou': best_val_miou}, CKPT_PATH)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"🛑 Early stopping tại epoch {epoch+1}")
                break

    # =====================================================================
    # 5. BÁO CÁO KẾT QUẢ CUỐI CÙNG (trên val.txt, do không có test.txt riêng)
    # =====================================================================
    ckpt = torch.load(CKPT_PATH, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    final_iou, final_miou, final_union = evaluate(model, val_loader)

    print("\n" + "="*60)
    print("TỔNG KẾT: REAL-ONLY (OFFICIAL SPLIT)")
    print("="*60)
    for cname, iou_val in zip(EVAL_CLASSES, final_iou):
        print(f"  {cname:20s}: {iou_val*100:.2f}%")
    print("-" * 60)
    print(f"🏆 Overall mIoU (val.txt) : {final_miou*100:.2f}%")
    print("="*60)

    np.save("/kaggle/working/results_realonly_officialsplit.npy",
            {"per_class_iou": final_iou, "mean_iou": final_miou, "classes": EVAL_CLASSES})

    del model, optimizer, scheduler, ckpt, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()



  🔥 Real-only protocol — dùng split gốc train.txt / val.txt

📊 Dataset: Train: 998 | Val: 246
⚠️  Lưu ý: split gốc không có test.txt riêng — val.txt vừa dùng để chọn best checkpoint, vừa dùng để báo cáo mIoU cuối cùng (không có tập test độc lập).


Epoch 1 | Val mIoU: 21.3%


Epoch 2 | Val mIoU: 32.9%


Epoch 3 | Val mIoU: 47.3%


Epoch 4 | Val mIoU: 46.6%


... (42 outputs left out)

Epoch 47 | Val mIoU: 87.5%


Epoch 48 | Val mIoU: 87.6%


Epoch 49 | Val mIoU: 87.5%


Epoch 50 | Val mIoU: 87.6%

TỔNG KẾT: REAL-ONLY (OFFICIAL SPLIT)
  Ground_vegetation   : 95.55%
  Trees               : 91.22%
  Dirt_ground         : 87.45%
  Road                : 95.34%
  Water               : 95.94%
  Empty               : 80.20%
  Static_car          : 91.77%
  Moving_car          : 83.52%
  People              : 67.51%
------------------------------------------------------------
🏆 Overall mIoU (val.txt) : 87.61%


In [3]:
import os, glob, gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# =====================================================================
# 1. CẤU HÌNH CƠ BẢN
# =====================================================================
ENCODER      = "resnet50"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE   = 4
EPOCHS       = 50
LR           = 1e-4
NUM_CLASSES  = 16
IGNORE_INDEX = 255
USE_AMP      = True
PATIENCE     = 10

WILD_UAV_OLD = "/kaggle/input/datasets/dinhh04/wilduav/WildUAV"
WILD_UAV_NEW = "/kaggle/input/datasets/phmnguynkimquyn/wuav-data"

# Split gốc của WildUAV (chỉ có train/val, KHÔNG có test.txt riêng)
TRAIN_TXT = "/kaggle/input/datasets/itmeagain77/wilduav-splits/train.txt"
VAL_TXT   = "/kaggle/input/datasets/itmeagain77/wilduav-splits/val.txt"

# Checkpoint pretrain trên Forest Inspection (seq3/6/9, sunny nadir)
PRETRAIN_CKPT = "/kaggle/input/datasets/itmeagain77/e50-nadir-pretrain/epoch_050_miou_0.7483.pth"

# =====================================================================
# 2. BẢNG MÀU, LUT & DATASET — GIỮ NGUYÊN NHƯ BẢN REAL-ONLY
# =====================================================================
WILDUAV_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65), (130, 76, 0),
    (152, 251, 152), (151, 126, 171), (0, 0, 255), (250, 150, 0), (115, 176, 195),
    (128, 64, 128), (255, 77, 228), (123, 123, 123), (255, 255, 255), (200, 0, 0), (0, 0, 0)
]
RGB_TO_WILDUAV_LUT = np.full((256, 256, 256), 15, dtype=np.uint8)
for cid, color in enumerate(WILDUAV_COLORS):
    RGB_TO_WILDUAV_LUT[color[0], color[1], color[2]] = cid

class WildUAVDataset(Dataset):
    def __init__(self, img_paths, label_paths, transform=None):
        self.img_paths   = img_paths
        self.label_paths = label_paths
        self.transform   = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img       = np.array(Image.open(self.img_paths[idx]).convert('RGB'))
        label_rgb = np.array(Image.open(self.label_paths[idx]).convert('RGB'))
        mask      = RGB_TO_WILDUAV_LUT[label_rgb[..., 0], label_rgb[..., 1], label_rgb[..., 2]]
        if self.transform:
            aug       = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
test_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

def read_split_file(txt_path):
    img_files, label_files = [], []
    with open(txt_path) as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    for line in lines:
        seq_name, fname = line.split('/')
        base_name = fname.split('.')[0]
        if seq_name == "seq00":
            img_dir = os.path.join(WILD_UAV_OLD, "mapping_set", f"{seq_name}_img+metadata", "img")
        else:
            img_dir = os.path.join(WILD_UAV_NEW, f"{seq_name}_imgmetadata.7z", f"{seq_name}_img+metadata.7z", "img")
        png_path = os.path.join(img_dir, f"{base_name}.png")
        jpg_path = os.path.join(img_dir, f"{base_name}.jpg")
        img_path = png_path if os.path.exists(png_path) else jpg_path
        label_path = os.path.join(WILD_UAV_OLD, "mapping_set", "semantic_extension", f"{seq_name}_semantic", "semantic", f"{base_name}.png")
        img_files.append(img_path)
        label_files.append(label_path)
    return img_files, label_files

def filter_valid_pairs(img_list, label_list):
    valid_imgs, valid_labels = [], []
    for img, lbl in zip(img_list, label_list):
        if os.path.exists(img) and os.path.exists(lbl):
            valid_imgs.append(img)
            valid_labels.append(lbl)
    return valid_imgs, valid_labels

EVAL_CLASSES = ['Ground_vegetation', 'Trees', 'Dirt_ground', 'Road', 'Water', 'Empty', 'Static_car', 'Moving_car', 'People']
NUM_EVAL = len(EVAL_CLASSES)

WILD_MAP_LUT = np.full((NUM_CLASSES,), 5, dtype=np.int64)  # default = Empty
WILD_MAP_LUT[0] = 0; WILD_MAP_LUT[5] = 0
WILD_MAP_LUT[1] = 1; WILD_MAP_LUT[2] = 1; WILD_MAP_LUT[3] = 1
WILD_MAP_LUT[4] = 2; WILD_MAP_LUT[10] = 3; WILD_MAP_LUT[11] = 3
WILD_MAP_LUT[7] = 4; WILD_MAP_LUT[15] = 5
WILD_MAP_LUT[12] = 6; WILD_MAP_LUT[13] = 7; WILD_MAP_LUT[14] = 8

def evaluate(model, loader):
    model.eval()
    cm = np.zeros((NUM_EVAL, NUM_EVAL), dtype=np.int64)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                preds = torch.argmax(model(imgs), dim=1).cpu().numpy()
            masks = masks.numpy()
            g_eval = WILD_MAP_LUT[masks]
            p_eval = WILD_MAP_LUT[preds]
            keep = (g_eval >= 0) & (g_eval < NUM_EVAL)
            cm += np.bincount(g_eval[keep] * NUM_EVAL + p_eval[keep], minlength=NUM_EVAL ** 2).reshape(NUM_EVAL, NUM_EVAL)

    inter = np.diag(cm)
    union = cm.sum(1) + cm.sum(0) - inter
    with np.errstate(divide='ignore', invalid='ignore'):
        iou = np.where(union > 0, inter / union, 0.0)
    return iou, iou.mean(), union

# =====================================================================
# 3. LOAD CHECKPOINT PRETRAIN (encoder + decoder, BỎ segmentation_head)
# =====================================================================
def load_pretrained_backbone(model, ckpt_path):
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    # Dò cấu trúc: có thể là state_dict thuần, hoặc dict bọc thêm key khác
    if isinstance(raw, dict) and 'model_state_dict' in raw:
        src_state = raw['model_state_dict']
        print("  [Checkpoint] Phát hiện dạng bọc {'model_state_dict': ...} — dùng key này.")
    elif isinstance(raw, dict) and all(isinstance(v, torch.Tensor) for v in raw.values()):
        src_state = raw
        print("  [Checkpoint] Phát hiện state_dict thuần (không bọc thêm key).")
    else:
        # Fallback: thử tìm bất kỳ key nào trông giống state_dict
        candidate_keys = [k for k, v in raw.items() if isinstance(v, dict)] if isinstance(raw, dict) else []
        raise ValueError(
            f"Không tự nhận diện được cấu trúc checkpoint. Các key cấp cao nhất: "
            f"{list(raw.keys()) if isinstance(raw, dict) else type(raw)}. "
            f"Cần bạn kiểm tra thủ công key nào chứa state_dict thật."
        )

    target_state = model.state_dict()

    # Chỉ giữ lại key thuộc encoder/decoder, loại segmentation_head (khác số lớp)
    # và loại bất kỳ key nào lệch shape so với model hiện tại (phòng hờ).
    load_state, skipped_shape, skipped_head = {}, [], []
    for k, v in src_state.items():
        if k.startswith("segmentation_head"):
            skipped_head.append(k)
            continue
        if k in target_state and target_state[k].shape == v.shape:
            load_state[k] = v
        else:
            skipped_shape.append(k)

    missing, unexpected = model.load_state_dict(load_state, strict=False)

    print(f"  [Checkpoint] Loaded {len(load_state)} tensor(s) từ encoder+decoder.")
    print(f"  [Checkpoint] Bỏ qua {len(skipped_head)} tensor(s) thuộc segmentation_head (khác số lớp): "
          f"{skipped_head[:4]}{' ...' if len(skipped_head) > 4 else ''}")
    if skipped_shape:
        print(f"  ⚠️  [Checkpoint] Bỏ qua thêm {len(skipped_shape)} tensor(s) do lệch shape ngoài dự kiến: "
              f"{skipped_shape[:4]}{' ...' if len(skipped_shape) > 4 else ''}")
    if missing:
        print(f"  ⚠️  [Checkpoint] {len(missing)} tensor(s) trong model hiện tại KHÔNG được khởi tạo từ pretrain "
              f"(sẽ giữ random init) — dự kiến gồm segmentation_head, kiểm tra list nếu thấy bất thường: "
              f"{missing[:4]}{' ...' if len(missing) > 4 else ''}")
    # unexpected nên rỗng vì ta đã tự lọc theo target_state ở trên; nếu không rỗng thì có gì đó bất thường
    if unexpected:
        print(f"  🛑 [Checkpoint] BẤT THƯỜNG — {len(unexpected)} tensor(s) unexpected: {unexpected[:4]}")

    return model

# =====================================================================
# 4. CHẠY 1 LẦN DUY NHẤT TRÊN SPLIT GỐC (TRANSFER, OFFICIAL SPLIT)
# =====================================================================
if __name__ == "__main__":
    print(f"\n{'='*60}")
    print(f"  🔥 Transfer protocol — pretrain synthetic → fine-tune split gốc train.txt/val.txt")
    print(f"{'='*60}\n")

    train_imgs_raw, train_labels_raw = read_split_file(TRAIN_TXT)
    val_imgs_raw,   val_labels_raw   = read_split_file(VAL_TXT)

    train_imgs, train_labels = filter_valid_pairs(train_imgs_raw, train_labels_raw)
    val_imgs,   val_labels   = filter_valid_pairs(val_imgs_raw,   val_labels_raw)
    print(f"📊 Dataset: Train: {len(train_imgs)} | Val: {len(val_imgs)}")
    print("⚠️  Lưu ý: split gốc không có test.txt riêng — val.txt vừa dùng để chọn best "
          "checkpoint, vừa dùng để báo cáo mIoU cuối cùng (không có tập test độc lập).")

    train_loader = DataLoader(WildUAVDataset(train_imgs, train_labels, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(WildUAVDataset(val_imgs,   val_labels, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # Khởi tạo model rồi nạp encoder+decoder từ checkpoint pretrain synthetic
    model = smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights=None, classes=NUM_CLASSES, activation=None)
    print("\n📦 Đang load checkpoint pretrain (seq3/6/9 sunny nadir)...")
    model = load_pretrained_backbone(model, PRETRAIN_CKPT)
    model = model.to(DEVICE)
    print()

    ce_loss   = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=IGNORE_INDEX)
    focal_loss = smp.losses.FocalLoss(mode='multiclass', ignore_index=IGNORE_INDEX)

    W_FOCAL = 1.0
    W_DICE  = 1.0
    W_CE    = 0.5
    
    def criterion(preds, targets):
        loss_ce = ce_loss(preds, targets)
        loss_dice = dice_loss(preds, targets)
        loss_focal = focal_loss(preds, targets)
        return (W_CE * loss_ce) + (W_DICE * loss_dice) + (W_FOCAL * loss_focal)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_val_miou = 0.0
    no_improve = 0
    CKPT_PATH = "/kaggle/working/best_transfer_officialsplit.pth"

    for epoch in range(EPOCHS):
        model.train()
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        scheduler.step()

        _, val_miou, _ = evaluate(model, val_loader)
        print(f"Epoch {epoch+1} | Val mIoU: {val_miou*100:.1f}%")

        if val_miou > best_val_miou:
            best_val_miou = val_miou
            no_improve = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_miou': best_val_miou}, CKPT_PATH)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"🛑 Early stopping tại epoch {epoch+1}")
                break

    # =====================================================================
    # 5. BÁO CÁO KẾT QUẢ CUỐI CÙNG (trên val.txt, do không có test.txt riêng)
    # =====================================================================
    ckpt = torch.load(CKPT_PATH, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    final_iou, final_miou, final_union = evaluate(model, val_loader)

    print("\n" + "="*60)
    print("TỔNG KẾT: TRANSFER (OFFICIAL SPLIT)")
    print("="*60)
    for cname, iou_val in zip(EVAL_CLASSES, final_iou):
        print(f"  {cname:20s}: {iou_val*100:.2f}%")
    print("-" * 60)
    print(f"🏆 Overall mIoU (val.txt) : {final_miou*100:.2f}%")
    print("="*60)

    np.save("/kaggle/working/results_transfer_officialsplit.npy",
            {"per_class_iou": final_iou, "mean_iou": final_miou, "classes": EVAL_CLASSES})

    del model, optimizer, scheduler, ckpt, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()


  🔥 Transfer protocol — pretrain synthetic → fine-tune split gốc train.txt/val.txt

📊 Dataset: Train: 998 | Val: 246
⚠️  Lưu ý: split gốc không có test.txt riêng — val.txt vừa dùng để chọn best checkpoint, vừa dùng để báo cáo mIoU cuối cùng (không có tập test độc lập).

📦 Đang load checkpoint pretrain (seq3/6/9 sunny nadir)...
  [Checkpoint] Phát hiện dạng bọc {'model_state_dict': ...} — dùng key này.
  [Checkpoint] Loaded 450 tensor(s) từ encoder+decoder.
  [Checkpoint] Bỏ qua 2 tensor(s) thuộc segmentation_head (khác số lớp): ['segmentation_head.0.weight', 'segmentation_head.0.bias']
  ⚠️  [Checkpoint] 2 tensor(s) trong model hiện tại KHÔNG được khởi tạo từ pretrain (sẽ giữ random init) — dự kiến gồm segmentation_head, kiểm tra list nếu thấy bất thường: ['segmentation_head.0.weight', 'segmentation_head.0.bias']



Epoch 1 | Val mIoU: 44.8%


Epoch 2 | Val mIoU: 48.9%


Epoch 3 | Val mIoU: 52.5%


Epoch 4 | Val mIoU: 62.8%


Epoch 5 | Val mIoU: 65.3%


... (41 outputs left out)

Epoch 47 | Val mIoU: 87.3%


Epoch 48 | Val mIoU: 87.5%


Epoch 49 | Val mIoU: 87.2%


Epoch 50 | Val mIoU: 87.3%

TỔNG KẾT: TRANSFER (OFFICIAL SPLIT)
  Ground_vegetation   : 95.62%
  Trees               : 91.35%
  Dirt_ground         : 87.03%
  Road                : 95.36%
  Water               : 96.04%
  Empty               : 79.50%
  Static_car          : 91.49%
  Moving_car          : 84.92%
  People              : 70.32%
------------------------------------------------------------
🏆 Overall mIoU (val.txt) : 87.96%
